# 置信度与候选重估计实验

读取冻结的11场景评价结果，汇总指标、风险曲线及局部误差诊断。方法和运行协议见 `doc/algorithm/confidence_extension.md`。运行前按说明生成实验产物，并选择SVP_env内核。

In [ ]:
import sys
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config/confidence_experiment.json").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

assert sys.version_info[:2] == (3, 11)
print("Kernel:", sys.version.split()[0], "OpenCV:", cv2.__version__)
artifacts = root / "artifacts/confidence_extension"
summary = json.loads((artifacts / "evaluation/summary.json").read_text())
frozen = json.loads((artifacts / "frozen.json").read_text())
print("Frozen candidate window:", frozen["candidate_block"])

## 冻结评价汇总
均值按场景等权。筛选的低误差必须与丢失的覆盖率一起看；修复保留原覆盖率。最终参数只在开发集选定。

In [ ]:
for method in ["baseline", "lrc_filter", "dps_filter", "refined"]:
    row = summary["summary"][method]
    print(f"{method:12s} MAE={row['mae_px']:.6f} px  RMSE={row['rmse_px']:.6f} px  bad2={row['bad2_fraction']:.2%}  coverage={row['coverage']:.2%}")
base = summary["summary"]["baseline"]
refined = summary["summary"]["refined"]
gain = 1-refined["mae_px"]/base["mae_px"]
print(f"MAE relative change: {gain:.2%}; predefined requirement: >=5%, coverage loss <=1 percentage point.")

In [ ]:
fractions = np.array([row["retained_fraction"] for row in summary["scenes"][0]["metrics"]["risk_curves"]["lrc"]])
fig, axes = plt.subplots(1,2,figsize=(13,4),layout="constrained")
for method in ["lrc", "dps"]:
    curves = [scene["metrics"]["risk_curves"][method] for scene in summary["scenes"]]
    axes[0].plot(fractions, np.mean([[r["confidence_mae_px"] for r in c] for c in curves],axis=0),marker="o",label=method.upper())
curves = [scene["metrics"]["risk_curves"]["lrc"] for scene in summary["scenes"]]
for key,label in [("random_mae_px","Random (seed 17)"),("oracle_mae_px","Oracle (diagnostic only)")]:
    axes[0].plot(fractions,np.mean([[r[key] for r in c] for c in curves],axis=0),linestyle="--",label=label)
axes[0].set(xlabel="Retained fraction of baseline valid predictions",ylabel="Scene-mean MAE (px)",title="Equal-count comparison")
axes[0].legend()
names = [s["scene"] for s in summary["scenes"]]
delta = [s["metrics"]["baseline"]["mae_px"]-s["metrics"]["refined"]["mae_px"] for s in summary["scenes"]]
axes[1].barh(names, delta, color=["seagreen" if d>0 else "firebrick" for d in delta])
axes[1].axvline(0,color="black",linewidth=.8)
axes[1].set(xlabel="Baseline MAE - refined MAE (px)",title="Positive: better; negative: worse")
fig.savefig(artifacts/"evaluation_overview.png",dpi=150)

## 局部替换误差诊断
以下区域用GT在实验完成后挑选，仅用于误差诊断，不参与推断或参数选择。绿色场景整体改善仍可能有失败像素，反之亦然。误差图越亮越差，灰色表示无效、未知或未评价；青色圆圈标出下方打印数值的像素。

In [ ]:
def show_region(scene_name, success=True):
    with np.load(artifacts/"evaluation"/scene_name/"arrays.npz") as data:
        arrays = {key:data[key] for key in data.files}
    b, f, t = arrays["baseline"], arrays["final"], arrays["truth"]
    known = arrays["reference"] & np.isfinite(t) & arrays["replaced"]
    improvement = np.zeros_like(b)
    improvement[known] = np.abs(b[known]-t[known])-np.abs(f[known]-t[known])
    evidence = np.maximum(improvement if success else -improvement, 0)
    if not np.any(evidence>0):
        raise ValueError("No illustrative example of the requested kind")
    window = cv2.boxFilter(evidence.astype(np.float32),-1,(61,61),normalize=False)
    y,x = np.unravel_index(np.argmax(window), window.shape)
    y0,y1=max(0,y-50),min(t.shape[0],y+51)
    x0,x1=max(0,x-50),min(t.shape[1],x+51)
    region=np.s_[y0:y1,x0:x1]
    selected=np.where(evidence[region]>0,evidence[region],0)
    py,px=np.unravel_index(np.argmax(selected),selected.shape)
    py,px=py+y0,px+x0
    print(scene_name,"pixel (x,y)=",(px,py),{k:float(arrays[k][py,px]) for k in ["baseline","candidate","final","truth","combined","candidate_confidence"]})
    print("Used thresholds: base<0.5; candidate>=0.5; candidate-base>=0.15")
    panels=[("Left image",cv2.cvtColor(arrays["left_bgr"],cv2.COLOR_BGR2RGB),None,None),
            ("Ground truth",t,"turbo",None),("Baseline",b,"turbo",None),("Combined confidence",arrays["combined"],"viridis",(0,1)),
            (f"Candidate (window {frozen['candidate_block']})",arrays["candidate"],"turbo",None),("Final",f,"turbo",None),
            ("Baseline abs error",np.where(arrays["reference"],np.abs(b-t),np.nan),"magma",(0,10)),
            ("Final abs error",np.where(arrays["reference"],np.abs(f-t),np.nan),"magma",(0,10))]
    fig,axes=plt.subplots(2,4,figsize=(14,7),layout="constrained")
    finite_t=t[region][np.isfinite(t[region])]
    lo,hi=np.percentile(finite_t,[1,99])
    for ax,(title,image,cmap,limits) in zip(axes.flat,panels):
        if cmap is None: ax.imshow(image[region])
        else:
            limits=limits if limits else (lo,hi)
            im=ax.imshow(np.ma.masked_invalid(image[region]),cmap=plt.get_cmap(cmap).with_extremes(bad="0.6"),vmin=limits[0],vmax=limits[1])
            fig.colorbar(im,ax=ax,shrink=.7)
        ax.scatter([px-x0],[py-y0],s=90,facecolors="none",edgecolors="cyan",linewidths=1.5)
        ax.set_title(title);ax.axis("off")
    fig.suptitle(f"{scene_name}: {'successful' if success else 'failed'} replacement example (post-hoc illustration)")
    fig.savefig(artifacts/("diagnostic_success.png" if success else "diagnostic_failure.png"),dpi=140)
    return fig

show_region("Jadeplant", True)
show_region("Shelves", False)
plt.show()